### 1. Imports

In [79]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

### 2. Settings

In [80]:
BACKGROUND_VALUE = 220.0
IMAGE_SIZE = (64, 64)
SEED = 5527


def set_seed(seed=5527):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

### 3. Load CSV

In [81]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [82]:
!find /content/drive/MyDrive -iname "*county*"

/content/drive/MyDrive/HW6_results/all_state_county_poverty.csv
/content/drive/MyDrive/HW6_results/county_image_and_poverty_rate.csv
/content/drive/MyDrive/HW6_results/county_image_and_poverty_rate_graybackground.csv
/content/drive/MyDrive/HW6_results/opt2_county_dark
/content/drive/MyDrive/HW6_results/__MACOSX/._opt2_county_dark
/content/drive/MyDrive/HW6_results/__MACOSX/opt2_county_dark
/content/drive/MyDrive/opt2_county_dark.zip
/content/drive/MyDrive/poverty_images/opt2_county_dark
/content/drive/MyDrive/poverty_images/__MACOSX/._opt2_county_dark
/content/drive/MyDrive/poverty_images/__MACOSX/opt2_county_dark


In [83]:
!find /content/drive/MyDrive -iname "*.zip"

/content/drive/MyDrive/opt2_county_dark.zip


In [84]:
!find /content/drive/MyDrive/poverty_images -iname "*.png" | head

/content/drive/MyDrive/poverty_images/opt2_county_dark/72_72059_Guayanilla.png
/content/drive/MyDrive/poverty_images/opt2_county_dark/48_48351_Newton.png
/content/drive/MyDrive/poverty_images/opt2_county_dark/48_48435_Sutton.png
/content/drive/MyDrive/poverty_images/opt2_county_dark/17_17103_Lee.png
/content/drive/MyDrive/poverty_images/opt2_county_dark/04_04007_Gila.png
/content/drive/MyDrive/poverty_images/opt2_county_dark/51_51027_Buchanan.png
/content/drive/MyDrive/poverty_images/opt2_county_dark/46_46053_Gregory.png
/content/drive/MyDrive/poverty_images/opt2_county_dark/40_40141_Tillman.png
/content/drive/MyDrive/poverty_images/opt2_county_dark/36_36091_Saratoga.png
/content/drive/MyDrive/poverty_images/opt2_county_dark/31_31017_Brown.png


In [85]:
BASE_DIR = "/content/drive/MyDrive/HW6_results"

IMAGE_DIR = "/content/drive/MyDrive/poverty_images/opt2_county_dark"

final_df = pd.read_csv(
    f"{BASE_DIR}/county_image_and_poverty_rate.csv",
    dtype={"county_id": str}
)

# rebuild image paths
final_df["image_path"] = final_df["image_path"].apply(
    lambda x:
    IMAGE_DIR + "/" + str(x).split("\\")[-1]
)

final_df = final_df[["image_path", "poverty_rate"]].copy()

final_df["poverty_rate"] = pd.to_numeric(
    final_df["poverty_rate"],
    errors="coerce"
)

final_df = final_df.dropna(subset=["poverty_rate"])

final_df = final_df[
    final_df["image_path"].str.lower().str.endswith(".png")
].copy()

# keep only existing files
final_df = final_df[
    final_df["image_path"].apply(lambda x: Path(str(x)).exists())
].reset_index(drop=True)

print("Final dataframe shape:", final_df.shape)

print(final_df.head())

print(final_df["poverty_rate"].describe())

Final dataframe shape: (3108, 2)
                                          image_path  poverty_rate
0  /content/drive/MyDrive/poverty_images/opt2_cou...          12.9
1  /content/drive/MyDrive/poverty_images/opt2_cou...          10.5
2  /content/drive/MyDrive/poverty_images/opt2_cou...          28.1
3  /content/drive/MyDrive/poverty_images/opt2_cou...          21.6
4  /content/drive/MyDrive/poverty_images/opt2_cou...          11.7
count    3108.000000
mean       14.109653
std         5.421886
min         3.800000
25%        10.400000
50%        13.200000
75%        16.800000
max        55.700000
Name: poverty_rate, dtype: float64


In [86]:
print(final_df["image_path"].iloc[0])

/content/drive/MyDrive/poverty_images/opt2_county_dark/01_01001_Autauga.png


### 4. Split

In [87]:
train_df, temp_df = train_test_split(
    final_df,
    test_size=0.30,
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED
)

print("\nSplit sizes")
print("Train:", train_df.shape)
print("Val:  ", val_df.shape)
print("Test: ", test_df.shape)


Split sizes
Train: (2175, 2)
Val:   (466, 2)
Test:  (467, 2)


### 5. Load Image

In [88]:
def load_gray_image(path, target_size=IMAGE_SIZE, background_value=BACKGROUND_VALUE):

    path = str(path)
    suffix = Path(path).suffix.lower()

    if suffix != ".png":
        raise ValueError(f"Expected PNG file, got {suffix}")

    pil_img = Image.open(path).convert("RGBA")

    arr = np.array(pil_img, dtype=np.float32)

    rgb = arr[..., :3]
    alpha = arr[..., 3] / 255.0

    gray = rgb.mean(axis=-1)

    # gray background composite
    img = alpha * gray + (1.0 - alpha) * background_value

    img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)

    img = np.clip(img, a_min=0, a_max=None)

    # IMPORTANT
    img = np.log1p(img)

    img_tensor = torch.tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

    _, _, h, w = img_tensor.shape

    target_h, target_w = target_size

    scale = min(target_h / h, target_w / w)

    new_h = max(1, int(round(h * scale)))
    new_w = max(1, int(round(w * scale)))

    img_tensor = F.interpolate(
        img_tensor,
        size=(new_h, new_w),
        mode="bilinear",
        align_corners=False
    )

    pad_h = target_h - new_h
    pad_w = target_w - new_w

    pad_top = pad_h // 2
    pad_bottom = pad_h - pad_top

    pad_left = pad_w // 2
    pad_right = pad_w - pad_left

    img_tensor = F.pad(
        img_tensor,
        (pad_left, pad_right, pad_top, pad_bottom),
        mode="constant",
        value=np.log1p(background_value)
    )

    img = img_tensor.squeeze(0).squeeze(0).numpy().astype(np.float32)

    return img


sample_img = load_gray_image(train_df.iloc[0]["image_path"])

print(sample_img.shape)
print(sample_img.min(), sample_img.max(), sample_img.mean())

(64, 64)
0.0 5.398163 2.1898167


### 6. Handcrafted Features

In [89]:
def extract_handcrafted_features(path, lit_threshold=0.10, bright_threshold=0.50):

    img = load_gray_image(path)

    pixels = img.reshape(-1)

    feats = {
        "mean_brightness": pixels.mean(),
        "max_brightness": pixels.max(),
        "std_brightness": pixels.std(),
        "median_brightness": np.median(pixels),
        "p25_brightness": np.percentile(pixels, 25),
        "p75_brightness": np.percentile(pixels, 75),
        "p90_brightness": np.percentile(pixels, 90),
        "p95_brightness": np.percentile(pixels, 95),

        "lit_pixel_share": np.mean(pixels > lit_threshold),
        "bright_pixel_share": np.mean(pixels > bright_threshold),

        "zero_pixel_share": np.mean(pixels <= 1e-6),
        "nonzero_pixel_share": np.mean(pixels > 1e-6)
    }

    return feats

### 7. Build Feature DF

In [90]:
def build_feature_df(df):

    feature_rows = []

    for _, row in df.iterrows():

        try:

            feats = extract_handcrafted_features(row["image_path"])

            feats["image_path"] = row["image_path"]
            feats["poverty_rate"] = row["poverty_rate"]

            feature_rows.append(feats)

        except Exception as e:

            print(f"Skipping corrupted image: {row['image_path']}")
            print(e)

    return pd.DataFrame(feature_rows)


train_feat_df = build_feature_df(train_df)
val_feat_df = build_feature_df(val_df)
test_feat_df = build_feature_df(test_df)

print(train_feat_df.head())

Skipping corrupted image: /content/drive/MyDrive/poverty_images/opt2_county_dark/47_47047_Fayette.png
cannot identify image file '/content/drive/MyDrive/poverty_images/opt2_county_dark/47_47047_Fayette.png'
   mean_brightness  max_brightness  std_brightness  median_brightness  \
0         2.189817        5.398163        2.586946           0.000000   
1         2.140176        5.398163        2.359837           0.691458   
2         3.038077        5.523515        1.996897           3.785506   
3         2.759254        5.448918        2.206552           3.663529   
4         2.106980        5.398163        2.500806           0.000000   

   p25_brightness  p75_brightness  p90_brightness  p95_brightness  \
0        0.000000        5.398163        5.398163        5.398163   
1        0.000000        5.398163        5.398163        5.398163   
2        0.291979        4.565535        5.398163        5.398163   
3        0.000000        4.711055        5.398163        5.398163   
4        

### 8. Feature Columns

In [91]:
feature_cols = [
    "mean_brightness",
    "max_brightness",
    "std_brightness",
    "median_brightness",
    "p25_brightness",
    "p75_brightness",
    "p90_brightness",
    "p95_brightness",
    "lit_pixel_share",
    "bright_pixel_share",
    "zero_pixel_share",
    "nonzero_pixel_share"
]

X_train = train_feat_df[feature_cols].copy()
y_train = train_feat_df["poverty_rate"].copy()

X_val = val_feat_df[feature_cols].copy()
y_val = val_feat_df["poverty_rate"].copy()

X_test = test_feat_df[feature_cols].copy()
y_test = test_feat_df["poverty_rate"].copy()

# 9. Metrics

In [92]:
def regression_metrics(y_true, y_pred, name="Model"):

    mse = mean_squared_error(y_true, y_pred)

    rmse = np.sqrt(mse)

    mae = mean_absolute_error(y_true, y_pred)

    r2 = r2_score(y_true, y_pred)

    print(f"\n{name}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE:  {mae:.4f}")
    print(f"R2:   {r2:.4f}")

    return {
        "rmse": rmse,
        "mae": mae,
        "r2": r2
    }

### 10. Ridge

In [93]:
ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])

ridge_model.fit(X_train, y_train)

val_pred_ridge = ridge_model.predict(X_val)
test_pred_ridge = ridge_model.predict(X_test)

ridge_val_metrics = regression_metrics(
    y_val,
    val_pred_ridge,
    name="Ridge Validation"
)

ridge_test_metrics = regression_metrics(
    y_test,
    test_pred_ridge,
    name="Ridge Test"
)


Ridge Validation
RMSE: 5.2273
MAE:  3.9703
R2:   0.0362

Ridge Test
RMSE: 5.7665
MAE:  4.3409
R2:   0.0202


###11. Random Forest

In [94]:
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=3,
    random_state=SEED,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

val_pred_rf = rf_model.predict(X_val)
test_pred_rf = rf_model.predict(X_test)

rf_val_metrics = regression_metrics(
    y_val,
    val_pred_rf,
    name="Random Forest Validation"
)

rf_test_metrics = regression_metrics(
    y_test,
    test_pred_rf,
    name="Random Forest Test"
)


Random Forest Validation
RMSE: 5.4515
MAE:  4.1632
R2:   -0.0482

Random Forest Test
RMSE: 5.8450
MAE:  4.3579
R2:   -0.0066


### 12. RF Importance

In [95]:
rf_importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

print("\nRandom Forest feature importance")
print(rf_importance_df)



Random Forest feature importance
                feature  importance
2        std_brightness    0.223996
0       mean_brightness    0.175073
9    bright_pixel_share    0.115547
3     median_brightness    0.087867
8       lit_pixel_share    0.080610
5        p75_brightness    0.079160
10     zero_pixel_share    0.076728
11  nonzero_pixel_share    0.061360
1        max_brightness    0.036511
6        p90_brightness    0.029875
7        p95_brightness    0.021661
4        p25_brightness    0.011614


### 13. CNN Dataset

In [96]:
y_train_mean = train_df["poverty_rate"].mean()
y_train_std = train_df["poverty_rate"].std()

print("\ny_train_mean:", y_train_mean)
print("y_train_std:", y_train_std)


def compute_mean_std(df):

    total_sum = 0.0
    total_sq_sum = 0.0
    total_pixels = 0

    for path in df["image_path"]:

        try:

            img = load_gray_image(path)

            total_sum += img.sum()
            total_sq_sum += (img ** 2).sum()

            total_pixels += img.size

        except Exception as e:

            print(f"Skipping corrupted image: {path}")
            print(e)

    mean = total_sum / total_pixels

    var = total_sq_sum / total_pixels - mean ** 2

    std = np.sqrt(max(var, 1e-8))

    return float(mean), float(std)


train_mean, train_std = compute_mean_std(train_df)

print("Train mean:", train_mean)
print("Train std:", train_std)

bad_file = "/content/drive/MyDrive/poverty_images/opt2_county_dark/47_47047_Fayette.png"

train_df = train_df[train_df["image_path"] != bad_file].reset_index(drop=True)

val_df = val_df[val_df["image_path"] != bad_file].reset_index(drop=True)

test_df = test_df[test_df["image_path"] != bad_file].reset_index(drop=True)

class CountyPovertyDataset1C(Dataset):

    def __init__(self, df, img_mean, img_std, y_mean, y_std):

        self.df = df.reset_index(drop=True).copy()

        self.img_mean = img_mean
        self.img_std = img_std

        self.y_mean = y_mean
        self.y_std = y_std

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        img = load_gray_image(row["image_path"])

        img = (img - self.img_mean) / (self.img_std + 1e-6)

        img = torch.tensor(img, dtype=torch.float32).unsqueeze(0)

        y = float(row["poverty_rate"])

        y = (y - self.y_mean) / (self.y_std + 1e-6)

        y = torch.tensor(y, dtype=torch.float32)

        return img, y


batch_size = 32

train_dataset = CountyPovertyDataset1C(
    train_df,
    train_mean,
    train_std,
    y_train_mean,
    y_train_std
)

val_dataset = CountyPovertyDataset1C(
    val_df,
    train_mean,
    train_std,
    y_train_mean,
    y_train_std
)

test_dataset = CountyPovertyDataset1C(
    test_df,
    train_mean,
    train_std,
    y_train_mean,
    y_train_std
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)



y_train_mean: 14.055586206896553
y_train_std: 5.350300683138612
Skipping corrupted image: /content/drive/MyDrive/poverty_images/opt2_county_dark/47_47047_Fayette.png
cannot identify image file '/content/drive/MyDrive/poverty_images/opt2_county_dark/47_47047_Fayette.png'
Train mean: 2.2447712421417236
Train std: 2.4287326335906982


### 14. CNN

In [97]:
class CNNRegressor1C(nn.Module):

    def __init__(self):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1,1))
        )

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, x, return_embedding=False):

        x = self.features(x)

        embedding = torch.flatten(x, start_dim=1)

        out = self.regressor(x).squeeze(1)

        if return_embedding:
            return out, embedding

        return out

### 15. Train CNN

In [98]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\nUsing device:", device)

model = CNNRegressor1C().to(device)

criterion = nn.SmoothL1Loss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)


def evaluate_regression(model, loader, criterion, device, y_mean, y_std):

    model.eval()

    all_preds_std = []
    all_targets_std = []

    running_loss = 0.0

    with torch.no_grad():

        for images, targets in loader:

            images = images.to(device)
            targets = targets.to(device)

            preds = model(images)

            loss = criterion(preds, targets)

            running_loss += loss.item() * images.size(0)

            all_preds_std.extend(preds.cpu().numpy())
            all_targets_std.extend(targets.cpu().numpy())

    all_preds_std = np.array(all_preds_std)
    all_targets_std = np.array(all_targets_std)

    all_preds = all_preds_std * (y_std + 1e-6) + y_mean
    all_targets = all_targets_std * (y_std + 1e-6) + y_mean

    mse = mean_squared_error(all_targets, all_preds)

    rmse = np.sqrt(mse)

    mae = mean_absolute_error(all_targets, all_preds)

    r2 = r2_score(all_targets, all_preds)

    avg_loss = running_loss / len(loader.dataset)

    return {
        "loss": avg_loss,
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "r2": r2
    }


def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    y_mean,
    y_std,
    epochs=50,
    patience=15
):

    best_val_loss = float("inf")

    best_state = None

    patience_counter = 0

    for epoch in range(1, epochs + 1):

        model.train()

        running_loss = 0.0

        for images, targets in train_loader:

            images = images.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()

            preds = model(images)

            loss = criterion(preds, targets)

            loss.backward()

            optimizer.step()

            running_loss += loss.item() * images.size(0)

        train_loss = running_loss / len(train_loader.dataset)

        val_metrics = evaluate_regression(
            model,
            val_loader,
            criterion,
            device,
            y_mean,
            y_std
        )

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val RMSE: {val_metrics['rmse']:.4f} | "
            f"Val MAE: {val_metrics['mae']:.4f} | "
            f"Val R2: {val_metrics['r2']:.4f}"
        )

        if val_metrics["loss"] < best_val_loss:

            best_val_loss = val_metrics["loss"]

            best_state = model.state_dict()

            patience_counter = 0

        else:

            patience_counter += 1

        if patience_counter >= patience:

            print("Early stopping triggered.")

            break

    if best_state is not None:

        model.load_state_dict(best_state)

    return model


model = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    y_mean=y_train_mean,
    y_std=y_train_std
)


test_metrics_cnn = evaluate_regression(
    model,
    test_loader,
    criterion,
    device,
    y_mean=y_train_mean,
    y_std=y_train_std
)

print("\nEnd-to-end CNN Test")
print(f"RMSE: {test_metrics_cnn['rmse']:.4f}")
print(f"MAE:  {test_metrics_cnn['mae']:.4f}")
print(f"R2:   {test_metrics_cnn['r2']:.4f}")


Using device: cuda
Epoch 01 | Train Loss: 0.3801 | Val Loss: 0.3671 | Val RMSE: 5.1639 | Val MAE: 3.9084 | Val R2: 0.0595
Epoch 02 | Train Loss: 0.3780 | Val Loss: 0.3679 | Val RMSE: 5.1979 | Val MAE: 3.8952 | Val R2: 0.0470
Epoch 03 | Train Loss: 0.3779 | Val Loss: 0.3645 | Val RMSE: 5.1824 | Val MAE: 3.8718 | Val R2: 0.0527
Epoch 04 | Train Loss: 0.3730 | Val Loss: 0.3609 | Val RMSE: 5.1533 | Val MAE: 3.8557 | Val R2: 0.0633
Epoch 05 | Train Loss: 0.3706 | Val Loss: 0.3626 | Val RMSE: 5.1898 | Val MAE: 3.8446 | Val R2: 0.0500
Epoch 06 | Train Loss: 0.3689 | Val Loss: 0.3916 | Val RMSE: 5.4896 | Val MAE: 3.9521 | Val R2: -0.0629
Epoch 07 | Train Loss: 0.3666 | Val Loss: 0.3723 | Val RMSE: 5.3140 | Val MAE: 3.8793 | Val R2: 0.0040
Epoch 08 | Train Loss: 0.3634 | Val Loss: 0.3604 | Val RMSE: 5.0929 | Val MAE: 3.8539 | Val R2: 0.0852
Epoch 09 | Train Loss: 0.3653 | Val Loss: 0.3537 | Val RMSE: 5.0962 | Val MAE: 3.7935 | Val R2: 0.0840
Epoch 10 | Train Loss: 0.3612 | Val Loss: 0.3614 | V

### 16. Embeddings

In [101]:
def extract_embeddings(model, loader, device, y_mean, y_std):

    model.eval()

    all_embeddings = []

    all_targets_std = []
    all_preds_std = []

    with torch.no_grad():

        for images, targets in loader:

            images = images.to(device)

            preds, embeddings = model(images, return_embedding=True)

            all_embeddings.append(embeddings.cpu().numpy())

            all_targets_std.extend(targets.numpy())
            all_preds_std.extend(preds.cpu().numpy())

    X = np.vstack(all_embeddings)

    y_std_arr = np.array(all_targets_std)

    y = y_std_arr * (y_std + 1e-6) + y_mean

    cnn_preds = np.array(all_preds_std) * (y_std + 1e-6) + y_mean

    return X, y, cnn_preds


X_train_emb, y_train_emb, train_cnn_pred = extract_embeddings(
    model,
    train_loader,
    device,
    y_train_mean,
    y_train_std
)

X_val_emb, y_val_emb, val_cnn_pred = extract_embeddings(
    model,
    val_loader,
    device,
    y_train_mean,
    y_train_std
)

X_test_emb, y_test_emb, test_cnn_pred = extract_embeddings(
    model,
    test_loader,
    device,
    y_train_mean,
    y_train_std
)

print("Embedding train shape:", X_train_emb.shape)
print("Embedding val shape:  ", X_val_emb.shape)
print("Embedding test shape: ", X_test_emb.shape)




Embedding train shape: (2174, 64)
Embedding val shape:   (466, 64)
Embedding test shape:  (467, 64)


### 17. Ridge on CNN Embeddings

In [102]:
ridge_emb_model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])

ridge_emb_model.fit(X_train_emb, y_train_emb)

val_pred_ridge_emb = ridge_emb_model.predict(X_val_emb)
test_pred_ridge_emb = ridge_emb_model.predict(X_test_emb)

ridge_emb_val_metrics = regression_metrics(
    y_val_emb,
    val_pred_ridge_emb,
    name="Ridge on CNN Embeddings Validation"
)

ridge_emb_test_metrics = regression_metrics(
    y_test_emb,
    test_pred_ridge_emb,
    name="Ridge on CNN Embeddings Test"
)


Ridge on CNN Embeddings Validation
RMSE: 5.2687
MAE:  3.9862
R2:   0.0209

Ridge on CNN Embeddings Test
RMSE: 5.5646
MAE:  4.1055
R2:   0.0877




### 18. Random Forest on CNN Embeddings


In [103]:
rf_emb_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=10,
    min_samples_leaf=3,
    random_state=SEED,
    n_jobs=-1
)

rf_emb_model.fit(X_train_emb, y_train_emb)

val_pred_rf_emb = rf_emb_model.predict(X_val_emb)
test_pred_rf_emb = rf_emb_model.predict(X_test_emb)

rf_emb_val_metrics = regression_metrics(
    y_val_emb,
    val_pred_rf_emb,
    name="RF on CNN Embeddings Validation"
)

rf_emb_test_metrics = regression_metrics(
    y_test_emb,
    test_pred_rf_emb,
    name="RF on CNN Embeddings Test"
)


RF on CNN Embeddings Validation
RMSE: 5.1649
MAE:  3.8630
R2:   0.0591

RF on CNN Embeddings Test
RMSE: 5.3980
MAE:  3.9737
R2:   0.1415


### 19. Final Comparison


In [104]:
hybrid_results_df = pd.DataFrame([
    {
        "model": "Ridge_handcrafted",
        "test_rmse": ridge_test_metrics["rmse"],
        "test_mae": ridge_test_metrics["mae"],
        "test_r2": ridge_test_metrics["r2"],
    },
    {
        "model": "RF_handcrafted",
        "test_rmse": rf_test_metrics["rmse"],
        "test_mae": rf_test_metrics["mae"],
        "test_r2": rf_test_metrics["r2"],
    },
    {
        "model": "End_to_end_CNN",
        "test_rmse": test_metrics_cnn["rmse"],
        "test_mae": test_metrics_cnn["mae"],
        "test_r2": test_metrics_cnn["r2"],
    },
    {
        "model": "Ridge_on_CNN_Embeddings",
        "test_rmse": ridge_emb_test_metrics["rmse"],
        "test_mae": ridge_emb_test_metrics["mae"],
        "test_r2": ridge_emb_test_metrics["r2"],
    },
    {
        "model": "RF_on_CNN_Embeddings",
        "test_rmse": rf_emb_test_metrics["rmse"],
        "test_mae": rf_emb_test_metrics["mae"],
        "test_r2": rf_emb_test_metrics["r2"],
    }
])

print("\nFinal model comparison")
print(hybrid_results_df)

hybrid_results_df.to_csv("hybrid_results_improved.csv", index=False)

rf_importance_df.to_csv("rf_feature_importance_improved.csv", index=False)

print("\nSaved hybrid_results_improved.csv")
print("Saved rf_feature_importance_improved.csv")


Final model comparison
                     model  test_rmse  test_mae   test_r2
0        Ridge_handcrafted   5.766501  4.340870  0.020239
1           RF_handcrafted   5.844997  4.357874 -0.006616
2           End_to_end_CNN   6.214338  4.432691 -0.137850
3  Ridge_on_CNN_Embeddings   5.564557  4.105460  0.087661
4     RF_on_CNN_Embeddings   5.398008  3.973676  0.141456

Saved hybrid_results_improved.csv
Saved rf_feature_importance_improved.csv
